# Acoustic and Hill tensors

From the stiffness $\mathbb{C}$ and a direction $\underline{\xi}$, the
**acoustic tensor** $\boldsymbol{K}=\underline{\xi}\cdot\mathbb{C}\cdot
\underline{\xi}$ controls wave propagation and, through its inverse, the
Green operator

$$
\mathbb{\Gamma}=\underline{\xi}\stackrel{s}{\otimes}\boldsymbol{K}^{-1}
                \stackrel{s}{\otimes}\underline{\xi},
\qquad
\mathbb{\Lambda}=\mathbb{C}:\mathbb{\Gamma}:\mathbb{C},
$$

the integrand of every Hill polarization tensor and the kernel of
stress-intensity-factor integrals on crack fronts.

The point of this tutorial is that the construction is written **once** and
runs unchanged on an isotropic and on a transversely isotropic stiffness — the
structured types of The Walpole basis doing the work.
Background: [mura1987](@cite), [hoenig1978](@cite).

In [1]:
using TensND
using LinearAlgebra
using SymPy

## Setting up

The direction $\underline{\xi}$ is taken as the position vector of a
spherical chart, so that $\underline{\xi}=\underline{e}^r$ up to its norm
and the angular dependence is explicit.

In [2]:
Cartesian = coorsys_cartesian(symbols("x y z", real = true))
𝐞₁, 𝐞₂, 𝐞₃ = unitvec(Cartesian)

Spherical = coorsys_spherical((symbols("θ ϕ", real = true)..., symbols("ξ", positive = true)))
θ, ϕ, ξ = getcoords(Spherical)
𝐞ᶿ, 𝐞ᵠ, 𝐞ʳ = unitvec(Spherical)

𝕀, 𝕁, 𝕂 = iso_projectors(Val(3), Val(Sym))
𝟏 = tens_Id2(Val(3), Val(Sym))
𝛏 = getOM(Spherical)

3-element TensND.TensRotated{1, 3, Sym{PyCall.PyObject}, Tensors.Vec{3, Sym{PyCall.PyObject}}}:
 0
 0
 ξ

## The isotropic case

With $\mathbb{C}=3\lambda\mathbb{J}+2\mu\mathbb{I}$:

In [3]:
λ = symbols("λ", real = true)
μ = symbols("μ", positive = true)
ℂ = 3λ * 𝕁 + 2μ * 𝕀

𝐊 = 𝛏 ⋅ ℂ ⋅ 𝛏

3×3 TensND.TensRotated{2, 3, Sym{PyCall.PyObject}, Tensors.SymmetricTensor{2, 3, Sym{PyCall.PyObject}, 6}}:
 μ*ξ^2      0              0
     0  μ*ξ^2              0
     0      0  ξ^2*(λ + 2*μ)

The acoustic tensor is transversely isotropic about $\underline{\xi}$: one
longitudinal eigenvalue and a doubly degenerate transverse one.

In [4]:
tsimplify(𝐊)

3×3 TensND.TensRotated{2, 3, Sym{PyCall.PyObject}, Tensors.SymmetricTensor{2, 3, Sym{PyCall.PyObject}, 6}}:
 μ*ξ^2      0              0
     0  μ*ξ^2              0
     0      0  ξ^2*(λ + 2*μ)

The Green operator and the Hill-type kernel:

In [5]:
ℾ = 𝛏 ⊗ˢ 𝐊^(-1) ⊗ˢ 𝛏
𝚲 = tsimplify(ℂ ⊡ ℾ ⊡ ℂ)

3×3×3×3 TensND.TensRotated{4, 3, Sym{PyCall.PyObject}, Tensors.SymmetricTensor{4, 3, Sym{PyCall.PyObject}, 36}}:
[:, :, 1, 1] =
 λ^2/(λ + 2*μ)              0  0
             0  λ^2/(λ + 2*μ)  0
             0              0  λ

[:, :, 2, 1] =
 0  0  0
 0  0  0
 0  0  0

[:, :, 3, 1] =
 0  0  μ
 0  0  0
 μ  0  0

[:, :, 1, 2] =
 0  0  0
 0  0  0
 0  0  0

[:, :, 2, 2] =
 λ^2/(λ + 2*μ)              0  0
             0  λ^2/(λ + 2*μ)  0
             0              0  λ

[:, :, 3, 2] =
 0  0  0
 0  0  μ
 0  μ  0

[:, :, 1, 3] =
 0  0  μ
 0  0  0
 μ  0  0

[:, :, 2, 3] =
 0  0  0
 0  0  μ
 0  μ  0

[:, :, 3, 3] =
 λ  0        0
 0  λ        0
 0  0  λ + 2⋅μ

It must reproduce the closed form

$$
\mathbb{\Lambda}=\frac{\lambda^{2}}{\lambda+2\mu}\,\boldsymbol{1}\otimes\boldsymbol{1}
+\frac{2\lambda\mu}{\lambda+2\mu}\bigl(\boldsymbol{1}\otimes\underline{e}^r\otimes\underline{e}^r
 +\underline{e}^r\otimes\underline{e}^r\otimes\boldsymbol{1}\bigr)
+4\mu\Bigl(\underline{e}^r\stackrel{s}{\otimes}\boldsymbol{1}\stackrel{s}{\otimes}\underline{e}^r
 -\frac{\lambda+\mu}{\lambda+2\mu}\,\underline{e}^r{}^{\otimes4}\Bigr)
$$

In [6]:
𝚲₂ = tsimplify(
    λ^2 / (λ + 2μ) * 𝟏 ⊗ 𝟏
        + 2λ * μ / (λ + 2μ) * (𝟏 ⊗ 𝐞ʳ ⊗ 𝐞ʳ + 𝐞ʳ ⊗ 𝐞ʳ ⊗ 𝟏)
        + 4μ * (𝐞ʳ ⊗ˢ 𝟏 ⊗ˢ 𝐞ʳ - (λ + μ) / (λ + 2μ) * 𝐞ʳ ⊗ 𝐞ʳ ⊗ 𝐞ʳ ⊗ 𝐞ʳ)
)

intrinsic(tsimplify(𝚲 - 𝚲₂), Spherical)

0


Identically zero.

## The transversely isotropic case

The same three lines, with a stiffness built by `tens_TI` instead.
Nothing in the construction changes: `⋅`, `inv` and `⊡` dispatch on the
structured type.

> **This replaces a legacy hand-rolled Walpole basis**
>
> An earlier research script re-defined the Walpole tensors and the TI
> constructors locally. Everything it needed is now in the library —
> `walpole_basis`, `tens_TI`, `tens_TI_eng`,
> `tens_TI_Hoenig` — so the local definitions are gone.

In [7]:
C₁₁₁₁, C₁₁₂₂, C₁₁₃₃, C₃₃₃₃, C₂₃₂₃ = symbols("C₁₁₁₁ C₁₁₂₂ C₁₁₃₃ C₃₃₃₃ C₂₃₂₃", positive = true)
n = 𝐞₃
ℂᵗⁱ = tens_TI(C₁₁₁₁, C₁₁₂₂, C₁₁₃₃, C₃₃₃₃, C₂₃₂₃, [Sym(0), Sym(0), Sym(1)])

typeof(ℂᵗⁱ), get_ℓ(ℂᵗⁱ)

(TensTI{4, Sym{PyCall.PyObject}, 5}, (C₃₃₃₃, C₁₁₁₁ + C₁₁₂₂, sqrt(2)*C₁₁₃₃, sqrt(2)*C₁₁₃₃, C₁₁₁₁ - C₁₁₂₂, 2*C₂₃₂₃))

The acoustic tensor along the symmetry axis. Taking
$\underline{\xi}=\underline{e}_3$ makes it diagonal, with the longitudinal
modulus $C_{3333}$ and the doubly degenerate shear modulus $C_{2323}$:

In [8]:
𝐊ᵃˣ = tsimplify(𝐞₃ ⋅ ℂᵗⁱ ⋅ 𝐞₃)
get_array(𝐊ᵃˣ)

3×3 Tensors.SymmetricTensor{2, 3, Sym{PyCall.PyObject}, 6}:
 C₂₃₂₃      0      0
     0  C₂₃₂₃      0
     0      0  C₃₃₃₃

In the isotropy plane, $\underline{\xi}=\underline{e}_1$, the three
eigenvalues are all distinct — the anisotropy is fully visible:

In [9]:
𝐊ᵗ = tsimplify(𝐞₁ ⋅ ℂᵗⁱ ⋅ 𝐞₁)
get_array(𝐊ᵗ)

3×3 Tensors.SymmetricTensor{2, 3, Sym{PyCall.PyObject}, 6}:
 C₁₁₁₁                  0      0
     0  C₁₁₁₁/2 - C₁₁₂₂/2      0
     0                  0  C₂₃₂₃

The two coincide only when the material is isotropic. Substituting the
isotropic relations $C_{1111}=C_{3333}=\lambda+2\mu$,
$C_{1122}=C_{1133}=\lambda$, $C_{2323}=\mu$:

In [10]:
iso_subs = Dict(
    C₁₁₁₁ => λ + 2μ, C₃₃₃₃ => λ + 2μ,
    C₁₁₂₂ => λ, C₁₁₃₃ => λ, C₂₃₂₃ => μ,
)
(tsimplify(subs.(get_array(𝐊ᵃˣ), iso_subs...)), tsimplify(subs.(get_array(𝐊ᵗ), iso_subs...)))

(Sym{PyCall.PyObject}[μ 0 0; 0 μ 0; 0 0 λ + 2*μ], Sym{PyCall.PyObject}[λ + 2*μ 0 0; 0 μ 0; 0 0 μ])

## The Green operator for the TI medium

Along the symmetry axis the acoustic tensor is diagonal, so its inverse is
immediate and $\mathbb{\Lambda}$ follows:

In [11]:
ℾᵗⁱ = 𝐞₃ ⊗ˢ inv(𝐊ᵃˣ) ⊗ˢ 𝐞₃
𝚲ᵗⁱ = tsimplify(ℂᵗⁱ ⊡ ℾᵗⁱ ⊡ ℂᵗⁱ)

get_array(𝚲ᵗⁱ)[3, 3, 3, 3]

C₃₃₃₃

The $3333$ component is $C_{3333}$ itself: along the symmetry axis the
Green operator exactly undoes the stiffness, as it must for a longitudinal
wave.

## Checking against the isotropic limit

Substituting the isotropic moduli into the TI result must reproduce the
isotropic $\mathbb{\Lambda}$ evaluated at $\underline{\xi}=\underline{e}_3$:

In [12]:
Λᵗⁱ_iso = tsimplify(subs.(get_array(𝚲ᵗⁱ), iso_subs...))
Λ_iso_axis = tsimplify(subs.(get_array(𝚲₂), θ => Sym(0), ϕ => Sym(0)))

tsimplify(Λᵗⁱ_iso - Λ_iso_axis)

3×3×3×3 Array{Sym{PyCall.PyObject}, 4}:
[:, :, 1, 1] =
 0  0  0
 0  0  0
 0  0  0

[:, :, 2, 1] =
 0  0  0
 0  0  0
 0  0  0

[:, :, 3, 1] =
 0  0  0
 0  0  0
 0  0  0

[:, :, 1, 2] =
 0  0  0
 0  0  0
 0  0  0

[:, :, 2, 2] =
 0  0  0
 0  0  0
 0  0  0

[:, :, 3, 2] =
 0  0  0
 0  0  0
 0  0  0

[:, :, 1, 3] =
 0  0  0
 0  0  0
 0  0  0

[:, :, 2, 3] =
 0  0  0
 0  0  0
 0  0  0

[:, :, 3, 3] =
 0  0  0
 0  0  0
 0  0  0

Zero: the transversely isotropic construction degenerates correctly.

---

*This notebook was generated using [Literate.jl](https://github.com/fredrikekre/Literate.jl).*